# Positional Encoding 심층 분석 - 실습 코드 2: Sinusoidal vs Learned vs RoPE 위치 인코딩 비교 실험

- Tutorial ID: `expand-positional-encoding`
- Tutorial: Positional Encoding 심층 분석
- Section ID: `expand-positional-encoding-code-2`
- Section: 실습 코드 2: Sinusoidal vs Learned vs RoPE 위치 인코딩 비교 실험

## 이 노트북을 읽는 법

이 노트북은 "정답 코드를 한 번 실행해보는 용도"가 아니라, **위치 인코딩(Positional Encoding)이라는 개념이 실제 텐서 연산으로 어떻게 구현되는지**를 한 줄씩 따라가며 이해하기 위한 실습 자료입니다.

### 학습 목표
1. Self-Attention이 왜 "토큰의 순서"를 스스로는 알 수 없는지 작은 실험으로 직접 확인합니다.
2. 위치 정보를 넣는 대표적인 방법 세 가지(Sinusoidal, Learned, RoPE)와, 보너스로 ALiBi까지 처음부터 직접 구현하고 비교합니다.
3. 각 방법이 "학습 때보다 더 긴 문장(외삽, extrapolation)"을 만났을 때 어떻게 반응하는지 실험으로 확인합니다.

### 읽는 순서
1. **Section 0** — 왜 위치 인코딩이 필요한지, 위치 정보가 없으면 어떤 문제가 생기는지 먼저 확인합니다.
2. **Section 1~3** — Sinusoidal → Learned → RoPE 순서로, 각 방법을 "아이디어 → 수식 → 코드 → 시각화"의 흐름으로 학습합니다.
3. **Section 4 (보너스)** — ALiBi를 추가로 살펴봅니다.
4. **Section 5** — 네 가지 방법을 한자리에 놓고 "더 긴 문장에도 잘 동작하는가?"를 비교 실험합니다.

### 코드를 읽을 때 체크할 것
- 텐서의 **shape**가 각 연산 후 어떻게 바뀌는지 (주석의 `# (B, N, D)` 같은 표기를 꼭 확인하세요)
- 숫자 하나하나를 외우기보다 **"정보가 어디서 어디로 이동하는지"**에 집중하세요
- 셀을 자유롭게 추가해서 변수 값을 바꿔보고 `print()`로 직접 확인해보는 것을 권장합니다

### 실행 환경 안내
이 노트북은 `torch`, `numpy`, `matplotlib`, `pandas`를 사용합니다. Google Colab에는 기본 설치되어 있고, 로컬 환경이라면 아래 명령으로 설치할 수 있습니다.

```bash
pip install torch numpy matplotlib pandas
```

그래프 안의 글자(제목/축 이름)는 한글 폰트가 없는 환경에서도 깨지지 않도록 영어로 표기했습니다. 그래프에 대한 설명은 그래프 주변의 한글 텍스트를 참고해주세요.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

torch.manual_seed(42)
print("torch:", torch.__version__)

## Section 0. 왜 위치 정보가 필요할까?

Self-Attention은 "지금 보고 있는 토큰(Query)과 비슷한 토큰(Key)을 찾아서, 그 값(Value)들을 가중합한다"는 연산입니다. 그런데 이 계산을 가만히 들여다보면 이상한 점이 있습니다.

> Attention 점수는 두 토큰의 **내용(벡터 값)** 만으로 계산되지, "이 토큰이 몇 번째인지"는 전혀 들여다보지 않습니다.

다시 말해, 입력 토큰들의 순서를 통째로 뒤섞어도 Self-Attention은 "뒤섞였다는 사실" 자체를 전혀 알아채지 못합니다. 이런 성질을 **순서 무관(permutation-equivariant)** 이라고 부릅니다 — 입력을 섞으면 출력도 그만큼 똑같이 섞일 뿐, 토큰 하나하나에 대한 계산 결과 자체는 동일합니다.

아래 코드에서 직접 확인해봅시다. 똑같은 토큰들을 순서만 뒤집어서 Attention에 넣었을 때, 정말 "결과도 그대로 뒤집히기만" 하는지 (= 모델이 순서를 전혀 신경쓰지 않는지) 검증합니다.

In [ ]:
# 위치 정보 없이 동작하는 아주 단순한 Self-Attention을 직접 구현해봅니다.
# (실제 Transformer는 Q,K,V를 만들 때 학습되는 가중치 행렬 W_Q, W_K, W_V를 곱하지만,
#  "토큰의 위치는 보지 않고 내용만 본다"는 핵심 성질은 동일하므로 여기서는 생략합니다.)
def simple_self_attention(x):
    # x: (batch, seq_len, d_model)
    d_model = x.shape[-1]
    # Query와 Key가 얼마나 비슷한지 점수로 계산 → (batch, seq_len, seq_len)
    scores = x @ x.transpose(-2, -1) / (d_model ** 0.5)
    attn_weights = torch.softmax(scores, dim=-1)
    # 비슷한 토큰일수록 더 높은 가중치로 Value(=x)를 섞어준다
    return attn_weights @ x

torch.manual_seed(0)

# "나는 / 사과 / 를 / 먹는다" 라는 4개 토큰짜리 문장이 있다고 생각해봅시다.
# 실제 단어 임베딩 대신, 각 토큰을 8차원 랜덤 벡터로 대신 표현합니다.
batch_size, seq_len, d_model_demo = 1, 4, 8
x = torch.randn(batch_size, seq_len, d_model_demo)

# (1) 원래 순서 그대로 Attention 적용: "나는 사과 를 먹는다"
out_original = simple_self_attention(x)

# (2) 토큰 순서를 완전히 뒤집어서 Attention 적용: "먹는다 를 사과 나는"
x_reversed = x.flip(dims=[1])
out_reversed = simple_self_attention(x_reversed)

print("입력 shape:", x.shape, " # (batch, seq_len, d_model)")
print()
print("[검증] '순서를 뒤집어 계산한 결과'를 다시 뒤집으면, '원래 순서로 계산한 결과'와 완전히 같을까?")
print("→", torch.allclose(out_original, out_reversed.flip(dims=[1]), atol=1e-6))

실행해보면 `True`가 출력됩니다. 즉, **토큰 순서를 뒤집어서 계산한 뒤 다시 뒤집으면, 애초에 뒤집지 않고 계산한 것과 완전히 똑같은 결과**가 나옵니다.

이것이 의미하는 바는 명확합니다. 지금의 Self-Attention은 "나는 사과를 먹는다"와 "먹는다 를 사과 나는"을 **각 토큰의 내용만 보면 똑같은 집합**으로 취급합니다. 출력 순서만 입력 순서를 그대로 따라갈 뿐, "이건 1번째 토큰이고 이건 4번째 토큰이다"라는 정보는 모델 어디에도 존재하지 않습니다.

하지만 자연어에서 순서는 의미를 결정짓는 핵심 정보입니다. ("개가 사람을 물었다" ≠ "사람이 개를 물었다")

그래서 토큰을 Attention에 넣기 **전에**, 또는 Attention을 계산하는 **도중에**, "이 토큰은 몇 번째 위치에 있다"는 정보를 직접 주입해야 합니다. 이것이 바로 **Positional Encoding(위치 인코딩)** 입니다. 지금부터 위치 정보를 주입하는 대표적인 방법들을 순서대로 살펴봅니다.

| 방법 | 핵심 아이디어 | 적용 시점 |
|---|---|---|
| **Sinusoidal** | 정해진 sin/cos 수식으로 위치 벡터를 만들어 더하기 | 토큰 임베딩에 더함 (Attention 이전) |
| **Learned** | 위치마다 학습되는 벡터를 하나씩 두고 더하기 | 토큰 임베딩에 더함 (Attention 이전) |
| **RoPE** | Query/Key 벡터를 위치에 비례한 각도로 회전시키기 | Attention 계산 도중 (Q, K에 적용) |
| *(보너스) ALiBi* | Attention 점수에 거리에 비례한 페널티(편향)를 더하기 | Attention 계산 도중 (score에 적용) |

## Section 1. Sinusoidal Positional Encoding

가장 먼저 등장한 방법으로, 2017년 Transformer 원 논문("Attention Is All You Need")에서 제안되었습니다.

### 핵심 아이디어
**위치마다 고유한 패턴을 가지는 벡터를, 학습 없이 고정된 sin/cos 수식으로 미리 만들어 둔다.**

위치 `pos`, 차원 인덱스 `i`에 대해 다음 수식으로 값을 채웁니다.

$$PE_{(pos,\ 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right), \qquad PE_{(pos,\ 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

말로 풀면, **짝수 번째 차원에는 sin, 홀수 번째 차원에는 cos** 값을 채우는데, 차원 인덱스 `i`가 커질수록 분모의 `10000^(2i/d_model)`이 커져서 **더 천천히 진동하는 파동**이 됩니다.

### 왜 하필 sin/cos이고, 왜 차원마다 다른 주파수를 쓸까?

시계를 떠올려보면 이해하기 쉽습니다. 시계는 초침·분침·시침 세 바늘이 **서로 다른 속도**로 돕니다.
- 초침(빠른 주파수)만 보면 "지금 몇 초인지"는 정확히 알 수 있지만, 몇 시간이 지났는지는 알 수 없습니다.
- 시침(느린 주파수)만 보면 대략적인 시간대는 알 수 있지만, 정확한 분/초는 알 수 없습니다.
- **세 바늘을 모두 합쳐서 봐야 비로소 정확한 시각이 결정**됩니다.

Sinusoidal PE도 같은 원리입니다. `d_model`개의 차원이 각각 다른 주파수로 진동하는 "바늘"들이고, 이 바늘들의 조합이 위치마다 유일한 패턴(지문처럼)을 만들어냅니다.
- **앞쪽 차원(i가 작음)**: 빠르게 진동 → 가까운 위치끼리도 값이 크게 달라짐 → 미세한 위치 차이를 구분
- **뒤쪽 차원(i가 큼)**: 느리게 진동 → 멀리 떨어진 위치까지 천천히 변함 → 거시적인 위치 차이를 구분

### 왜 굳이 sin과 cos을 "함께" 쓸까?
sin 값만 보면, 값이 같은 두 각도(예: 30°와 150°)를 구분할 수 없습니다. cos까지 함께 보면 `(sin θ, cos θ)` 쌍이 원 위의 한 점을 유일하게 결정하므로 이런 모호함이 사라집니다.

In [ ]:
class SinusoidalPE(nn.Module):
    """
    원조 Transformer의 위치 인코딩.
    학습되는 파라미터가 하나도 없고, 정해진 수식으로 위치 벡터를 만들어서 토큰 임베딩에 "더하기"만 합니다.
    """

    def __init__(self, d_model, max_len=5000):
        """
        d_model : 임베딩(토큰 벡터) 차원 수. 위치 벡터도 같은 차원으로 만들어서 그대로 더할 수 있게 합니다.
        max_len : 미리 계산해 둘 최대 위치 개수. (수식 자체에는 위치 제한이 없지만, 매번 다시
                  계산하지 않도록 넉넉히 미리 만들어서 캐싱해 두는 것입니다.)
        """
        super().__init__()

        # pe[pos, dim] 에 들어갈 값을 담을 빈 표(table)를 0으로 초기화
        pe = torch.zeros(max_len, d_model)  # shape: (max_len, d_model)

        # position = [[0], [1], [2], ..., [max_len-1]]  → shape: (max_len, 1)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        # 차원마다 다른 "진동 속도(주파수)"를 계산합니다.
        # i = 0, 2, 4, ..., d_model-2 에 대해 10000^(-i/d_model) 값을 구하는 것과 동일합니다.
        # (큰 수 10000^(...)을 직접 거듭제곱하면 수치적으로 불안정해지기 쉬워서,
        #  exp(log(...)) 형태로 풀어 써서 안전하게 계산합니다.)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )  # shape: (d_model/2,)  예) d_model=128 이면 64개의 서로 다른 주파수

        # 짝수 인덱스 차원(0,2,4,...)에는 sin, 홀수 인덱스 차원(1,3,5,...)에는 cos을 채웁니다.
        # position(max_len,1) * div_term(d_model/2,) → 브로드캐스팅으로 (max_len, d_model/2) 결과
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # 배치 차원을 하나 추가해서 (1, max_len, d_model) 형태로 만들어 둡니다.
        # → forward에서 (batch, seq_len, d_model) 입력에 그대로 더할 수 있도록 하기 위함입니다.
        #
        # register_buffer: 학습되는 파라미터(nn.Parameter)는 아니지만, model.to(device)나
        # state_dict 저장/로드 시 함께 따라다니도록 등록하는 PyTorch의 표준적인 방법입니다.
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        # 미리 만들어둔 위치 벡터 표(self.pe)에서 입력 길이(seq_len)만큼만 잘라서 더해줍니다.
        return x + self.pe[:, :x.size(1)]

In [ ]:
# 아주 작은 예시로 실제 값을 직접 눈으로 확인해봅니다 (d_model=6 처럼 작은 값을 쓰면 표 전체가 한눈에 보입니다)
toy_pe = SinusoidalPE(d_model=6, max_len=4)
print("d_model=6, position 0~3 일 때의 위치 벡터 (행=위치, 열=차원 0~5):")
print(np.round(toy_pe.pe[0].numpy(), 4))
print()
print("→ position=0(첫 번째 행)을 보면 sin(0)=0, cos(0)=1 이 반복되는 패턴이 보입니다.")
print("→ 0,1번 차원(앞쪽, 빠른 주파수)은 위치가 하나 바뀔 때마다 값이 크게 변하고,")
print("  4,5번 차원(뒤쪽, 느린 주파수)은 상대적으로 천천히 변하는 것을 확인해보세요.")

In [ ]:
# 앞으로 이 노트북 전체에서 공통으로 사용할 기본 하이퍼파라미터입니다.
d_model = 128   # 임베딩(토큰 벡터) 차원
n_heads = 4     # Multi-Head Attention의 head 개수 (RoPE, ALiBi에서 사용)

sin_pe = SinusoidalPE(d_model=d_model, max_len=5000)
dummy_x = torch.zeros(1, 50, d_model)
pe_output = sin_pe(dummy_x)
print("PE 적용 결과 shape:", pe_output.shape, " # (batch=1, seq_len=50, d_model=128)")

In [ ]:
# 위치(세로축) x 차원(가로축)을 히트맵으로 그려서, sin/cos 패턴이 만드는 "줄무늬"를 한눈에 봅니다.
plt.figure(figsize=(10, 5))
plt.imshow(sin_pe.pe[0, :100, :].numpy(), cmap='RdBu', aspect='auto')
plt.colorbar(label='value')
plt.xlabel('dimension index')
plt.ylabel('position')
plt.title('Sinusoidal Positional Encoding (position 0-99, dim 0-127)')
plt.tight_layout()
plt.show()

위 히트맵에서 **왼쪽(차원 인덱스가 작은 곳)은 세로 줄무늬가 촘촘하고**, **오른쪽으로 갈수록 줄무늬 간격이 넓어지는** 것을 볼 수 있습니다. "앞쪽 차원은 빠르게, 뒤쪽 차원은 느리게 진동한다"는 것을 시각적으로 보여주는 그림입니다.

아래에서는 몇 개 차원만 따로 뽑아서, 위치가 바뀔 때 값이 어떻게 변하는지 선 그래프로 비교해봅니다.

In [ ]:
positions_to_plot = np.arange(0, 100)
dims_to_plot = [0, 1, 20, 21, 100, 101]  # sin,cos 쌍을 기준으로 앞 / 중간 / 뒤쪽 차원을 골라봄

plt.figure(figsize=(10, 4))
for d in dims_to_plot:
    plt.plot(positions_to_plot, sin_pe.pe[0, :100, d].numpy(), label=f'dim {d}')
plt.xlabel('position')
plt.ylabel('value')
plt.title('Different dimensions oscillate at different frequencies')
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

그래프를 보면 `dim 0, 1`(앞쪽)은 위치가 몇 칸만 바뀌어도 값이 빠르게 오르내리지만, `dim 100, 101`(뒤쪽)은 100칸이 지나도 거의 변화가 없을 정도로 천천히 움직입니다. 이렇게 여러 주파수를 겹쳐 쌓았기 때문에 모든 위치가 서로 다른 고유한 패턴(벡터)을 가지게 됩니다.

### Sinusoidal PE의 중요한 성질: "상대적 위치"도 표현할 수 있다

신기하게도 이 수식은 단순히 "절대 위치"만 표현하는 게 아닙니다. 삼각함수의 덧셈정리 덕분에, **위치 `pos+k`의 인코딩 벡터는 위치 `pos`의 인코딩 벡터를 (k에 의해서만 정해지는) 어떤 고정된 회전 행렬로 선형 변환한 것과 같습니다.** 즉 모델이 "고정된 변환 하나"만 학습하면, 그 변환을 이용해 "k칸 떨어진 위치"라는 상대적 관계까지 알아낼 여지가 생깁니다.

> 💡 **미리 보기**: 이 "회전으로 상대적 위치 관계를 표현한다"는 아이디어를, 더하기가 아니라 Q/K 벡터 자체를 회전시키는 방식으로 더 직접적으로 밀어붙인 방법이 바로 잠시 후 배울 **RoPE**입니다.

### Sinusoidal PE 요약
- ✅ 학습 파라미터 없음 (수식으로 고정)
- ✅ 이론적으로는 어떤 길이의 문장에도 적용 가능 (외삽 가능)
- ⚠️ 다만 학습 때 본 적 없는 먼 위치에 대해서는 실제 성능이 떨어지는 경우가 많음 (수식 자체는 계산되지만, 모델이 그 패턴의 "의미"를 학습 데이터에서 본 적이 없기 때문)

## Section 2. Learned Positional Embedding

BERT, GPT-2 등 많은 모델이 사용한 방식입니다. 아이디어는 Sinusoidal보다 오히려 더 단순합니다.

### 핵심 아이디어
**"단어 임베딩"을 만들 때처럼, "위치 임베딩"도 그냥 학습시키자.**

토큰(단어) 하나하나에 `nn.Embedding(vocab_size, d_model)`로 학습 가능한 벡터를 하나씩 배정하듯이, **위치 0번, 1번, 2번, ... 에도 학습 가능한 벡터를 하나씩** 배정합니다. 즉 `nn.Embedding(max_len, d_model)`을 그대로 사용하고, 입력으로 "토큰 id" 대신 "위치 id(0,1,2,...)"를 넣을 뿐입니다.

### Sinusoidal과 무엇이 다를까?

| | Sinusoidal | Learned |
|---|---|---|
| 값이 어떻게 정해지나 | 고정된 sin/cos 수식 | 처음엔 무작위, 학습하면서 값이 바뀜(역전파로 업데이트) |
| 학습 파라미터 | 없음 | `max_len × d_model`개 |
| 학습 때 보지 못한 위치(예: max_len=100인데 200번째 토큰)를 만나면 | 수식으로 계산 가능 | **해당 위치의 벡터 자체가 존재하지 않아 에러 발생** |

마지막 줄이 Learned 방식의 가장 큰 약점입니다. `nn.Embedding(max_len, ...)`은 `max_len`을 만들 때 미리 정해야 하는 "표(table)"이기 때문에, 그보다 긴 위치 인덱스가 들어오면 아예 조회할 행이 없어서 에러가 납니다.

In [ ]:
class LearnedPE(nn.Module):
    """
    BERT, GPT-2 스타일의 학습 가능한 위치 임베딩.
    단어 임베딩과 똑같은 구조(nn.Embedding)를, "토큰 id" 대신 "위치 id"를 조회하는 데 사용합니다.
    """

    def __init__(self, d_model, max_len=512):
        super().__init__()
        # max_len개의 위치 각각에 대해 학습 가능한 d_model 차원 벡터를 준비합니다.
        # 처음에는 (단어 임베딩처럼) 무작위 값으로 초기화되고, 학습 데이터를 통해 점점 의미를 갖게 됩니다.
        self.embedding = nn.Embedding(max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        seq_len = x.size(1)

        # 0, 1, 2, ..., seq_len-1 이라는 "위치 id" 시퀀스를 만듭니다.
        # (단어를 vocab id로 바꾸는 것처럼, 위치를 position id로 바꾸는 것뿐입니다)
        positions = torch.arange(seq_len, device=x.device)  # shape: (seq_len,)

        # 위치 id를 임베딩 벡터로 변환 → (seq_len, d_model)
        # 이를 입력 x: (batch, seq_len, d_model) 에 브로드캐스팅으로 더해줍니다.
        return x + self.embedding(positions)

In [ ]:
learned_pe = LearnedPE(d_model=d_model, max_len=100)

# max_len(100) 이내의 위치는 문제없이 동작합니다.
ok_input = torch.zeros(1, 50, d_model)
print("길이 50 입력 (max_len=100 이내):", learned_pe(ok_input).shape, "→ 정상 동작 ✅")

# max_len(100)을 넘어서는 위치를 조회하면 어떻게 될까요? 직접 에러를 발생시켜 확인해봅니다.
too_long_input = torch.zeros(1, 150, d_model)
try:
    learned_pe(too_long_input)
except Exception as e:
    print()
    print("길이 150 입력 (max_len=100 초과) → 에러 발생 ❌")
    print(f"   에러 메시지: {e}")

In [ ]:
# 학습 전(=무작위 초기화 상태)의 Learned PE를 시각화해봅니다.
# Sinusoidal과 달리, 아직 아무런 규칙적인 패턴이 없는 "노이즈"처럼 보일 것입니다.
with torch.no_grad():
    learned_table = learned_pe.embedding.weight[:100].numpy()  # (100, d_model)

plt.figure(figsize=(10, 5))
plt.imshow(learned_table, cmap='RdBu', aspect='auto')
plt.colorbar(label='value')
plt.xlabel('dimension index')
plt.ylabel('position')
plt.title('Learned Positional Embedding - BEFORE training (random init)')
plt.tight_layout()
plt.show()

print("Sinusoidal 히트맵과 비교해보세요: 줄무늬 같은 규칙적인 구조가 전혀 없습니다.")
print("Learned PE는 '학습을 통해서만' 이런 무작위 벡터들이 위치 관계를 표현하도록 의미를 갖게 됩니다.")

### Learned PE 요약
- ✅ 데이터에 맞춰 유연하게 최적화될 수 있음 (꼭 sin/cos 형태일 필요가 없으므로, 데이터 특성에 더 잘 맞는 패턴을 학습할 가능성)
- ✅ 구현이 매우 단순함 (임베딩 룩업 한 줄)
- ❌ `max_len`을 넘는 위치는 원천적으로 표현 불가능 (외삽 불가능)
- ❌ 위치 임베딩 자체도 데이터로부터 "학습"되어야 하므로, 등장 빈도가 낮은 먼 위치(테이블 끝부분)는 충분히 학습되지 않을 수 있음

## Section 3. RoPE (Rotary Positional Embedding)

LLaMA, GPT-NeoX, PaLM, Qwen 등 최근 대형 언어모델 대부분이 사용하는 방식입니다. (논문: Su et al., *RoFormer*, 2021)

지금까지 본 두 방법(Sinusoidal, Learned)은 공통점이 있습니다. **"위치 벡터를 만들어서 토큰 임베딩에 더하기"**. 토큰이 "자신의 절대 위치"를 직접 기억하게 만드는 방식입니다.

RoPE는 완전히 다른 전략을 씁니다.

### 핵심 아이디어
**위치 정보를 "더하지" 않고, Query·Key 벡터를 위치에 비례한 각도만큼 "회전"시킨다.**

더하기가 아니라 회전이라는 게 무슨 뜻인지, 가장 작은 단위인 **2차원 벡터 하나**로 먼저 살펴봅시다.

### 2차원에서 먼저 이해하기
벡터 $(x_1, x_2)$를 2차원 평면 위의 한 점이라고 생각하면, 이 점을 원점을 중심으로 각도 $\theta$만큼 회전시키는 공식은 다음과 같습니다 (학교 수학에서 배우는 회전 변환입니다).

$$\begin{pmatrix} x_1' \\ x_2' \end{pmatrix} = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix}$$

RoPE의 아이디어는 **위치 `m`에 있는 벡터를 각도 `m × θ`만큼 회전**시키자는 것입니다. 위치가 0이면 회전 안 함, 위치가 1이면 `θ`만큼, 위치가 2면 `2θ`만큼... 점점 더 많이 돌립니다.

### 왜 "회전"이면 상대적 위치가 자연스럽게 나올까?
여기가 RoPE의 핵심이자 가장 영리한 부분입니다. Attention 점수는 Query와 Key의 **내적(dot product)**으로 계산됩니다. 회전 변환은 "각도를 보존하는" 변환이라서, **두 벡터를 각각 회전시킨 뒤 내적을 구하면, 그 결과는 (개별 회전 각도가 아니라) "두 회전 각도의 차이"에만 의존**하게 됩니다.

말로 풀면, 위치 `m`의 Query를 `m×θ`만큼, 위치 `n`의 Key를 `n×θ`만큼 회전시킨 뒤 내적을 구하면 → 그 결과는 항상 **`(n-m)×θ`, 즉 둘 사이의 거리에만** 의존합니다. `m=5, n=2`(거리 3)이든 `m=105, n=102`(거리 3)이든 **내적 값이 똑같다**는 뜻입니다.

→ 절대 위치(5번째, 105번째)를 더해주는 게 아니라, **"몇 칸 떨어져 있는가"라는 상대 거리를 attention score 자체에 자동으로 녹여내는 것**입니다. 이것이 RoPE가 "상대적 위치 인코딩(relative positional encoding)"으로 분류되는 이유입니다.

잠시 후 코드로 이 성질을 직접 숫자로 확인해보겠습니다. 먼저 2차원을 넘어 실제 `d_model` 차원 전체로 어떻게 일반화하는지 봅시다.

### 차원 전체로 확장하기

실제 벡터는 2차원이 아니라 `d_model`(예: 128)차원입니다. RoPE는 이걸 **2개씩 짝지어서**, 각 쌍마다 독립적으로 위에서 본 2차원 회전을 적용합니다.

$$(x_0, x_1),\ (x_2, x_3),\ (x_4, x_5),\ \ldots,\ (x_{d-2}, x_{d-1})$$

이렇게 `d_model/2`개의 쌍이 생기고, **각 쌍마다 서로 다른 회전 속도(각도) $\theta_i$**를 사용합니다.

$$\theta_i = 10000^{-2i/d_{model}} \quad (i = 0, 1, \ldots, d_{model}/2 - 1)$$

눈치채셨겠지만, 이 $\theta_i$를 정의하는 수식은 Section 1의 Sinusoidal PE에서 본 주파수 수식과 **사실상 같은 형태**입니다! 앞쪽 차원 쌍(i가 작음)은 빠르게 회전하고, 뒤쪽 차원 쌍(i가 큼)은 느리게 회전합니다 — 시계의 초침/분침/시침 비유가 여기서도 그대로 적용됩니다. 차이는 "이 값을 더하느냐(Sinusoidal) vs Q,K를 이 각도로 회전시키느냐(RoPE)" 뿐입니다.

In [ ]:
class RoPE(nn.Module):
    """
    RoPE: Rotary Positional Embedding (Su et al., 2021)
    Query/Key 벡터를 "위치 x 각도"만큼 회전시켜서, Attention score에 상대적 위치 정보가
    자연스럽게 녹아들도록 만드는 모듈입니다.
    """

    def __init__(self, d_model, max_len=5000, base=10000):
        """
        d_model : 회전을 적용할 벡터의 차원 (Attention head 하나의 차원, 즉 d_k와 같습니다)
        max_len : 미리 cos/sin 값을 계산해서 캐싱해 둘 최대 위치 개수
        base    : 회전 속도를 정하는 기준 상수 (Sinusoidal PE의 10000과 동일한 역할)
        """
        super().__init__()
        assert d_model % 2 == 0, "RoPE는 차원을 2개씩 짝지어 회전시키므로 d_model은 짝수여야 합니다."

        # i = 0, 1, ..., d_model/2 - 1 에 대해 회전 속도 theta_i = base^(-2i/d_model) 계산
        # → 앞쪽 차원 쌍일수록 theta가 커서 "빠르게" 회전(위치 변화에 민감),
        #   뒤쪽 차원 쌍일수록 theta가 작아서 "천천히" 회전(긴 거리까지 구분)
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)  # shape: (d_model/2,)

        # 위치 0 ~ max_len-1 전체에 대해 "위치 x theta_i" 각도를 미리 계산해서 cos, sin 표로 캐싱해둡니다.
        # (Attention을 계산할 때마다 다시 계산하지 않기 위한 효율화입니다)
        positions = torch.arange(max_len).float()  # (max_len,)
        # position(max_len,1) * inv_freq(1,d_model/2) → 브로드캐스팅으로 (max_len, d_model/2)
        # (Section 1의 Sinusoidal PE에서 했던 것과 정확히 같은 broadcasting 패턴입니다)
        freqs = positions.unsqueeze(1) * inv_freq.unsqueeze(0)
        self.register_buffer('cos_cached', freqs.cos())
        self.register_buffer('sin_cached', freqs.sin())

    def apply_rotation(self, x, positions):
        """
        x         : (..., d_model) 회전시킬 벡터 (Q 또는 K)
        positions : (seq_len,) 각 위치의 정수 인덱스
        """
        cos = self.cos_cached[positions]  # (seq_len, d_model/2)
        sin = self.sin_cached[positions]  # (seq_len, d_model/2)

        # 차원을 2개씩 짝지어 "짝수 번째 묶음(x_even)"과 "홀수 번째 묶음(x_odd)"으로 나눕니다.
        # 예) d_model=6 이면 x=[x0,x1,x2,x3,x4,x5] → x_even=[x0,x2,x4], x_odd=[x1,x3,x5]
        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        # 앞서 본 2D 회전 공식을 모든 쌍에 동시에(브로드캐스팅으로) 적용합니다.
        #   x_even' = x_even * cos - x_odd * sin
        #   x_odd'  = x_even * sin + x_odd * cos
        rotated_even = x_even * cos - x_odd * sin
        rotated_odd = x_even * sin + x_odd * cos

        # 회전시킨 두 그룹을 다시 (x0', x1', x2', x3', ...) 순서로 끼워 맞춰 원래 모양으로 복원합니다.
        x_rotated = torch.stack([rotated_even, rotated_odd], dim=-1).flatten(start_dim=-2)
        return x_rotated

    def forward(self, x):
        # x: (..., seq_len, d_model) — 보통은 (batch, n_heads, seq_len, d_k) 형태로 들어옵니다.
        # 0번째 토큰부터 순서대로 위치가 매겨진다고 가정하고 회전을 적용합니다.
        seq_len = x.shape[-2]
        positions = torch.arange(seq_len, device=x.device)
        return self.apply_rotation(x, positions)

In [ ]:
# 2D 토이 예제: 벡터 하나가 위치에 따라 어떻게 회전하는지 직접 눈으로 봅니다.
toy_rope = RoPE(d_model=2, max_len=20)  # 2차원이므로 회전 속도(theta)는 1개뿐 (theta_0 = base^0 = 1)

base_vector = torch.tensor([1.0, 0.0])  # 회전시킬 기준 벡터: x축 방향의 단위 벡터
positions_to_show = list(range(6))      # 0,1,2,3,4,5 (위치가 1 늘어날 때마다 1라디안씩 회전)

plt.figure(figsize=(5, 5))
origin = np.zeros(2)
colors = plt.cm.viridis(np.linspace(0, 1, len(positions_to_show)))
for pos, color in zip(positions_to_show, colors):
    rotated = toy_rope.apply_rotation(base_vector, torch.tensor([pos]))[0].numpy()
    plt.quiver(*origin, *rotated, angles='xy', scale_units='xy', scale=1, color=color)
    plt.text(rotated[0] * 1.15, rotated[1] * 1.15, f'pos={pos}', fontsize=9, color=color, ha='center')

plt.xlim(-1.4, 1.4)
plt.ylim(-1.4, 1.4)
plt.gca().set_aspect('equal')
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.title('Same vector rotated by different positions (RoPE, d=2)')
plt.tight_layout()
plt.show()

print("같은 벡터 (1, 0)이 position 값에 따라 점점 더 많이 회전하는 것을 볼 수 있습니다.")
print("이 토이 예제에서는 position이 1 늘어날 때마다 정확히 1라디안(theta=1)씩 회전합니다.")

In [ ]:
# 핵심 성질 검증: "회전된 두 벡터의 내적은 절대 위치가 아니라 상대 거리에만 의존한다"
torch.manual_seed(7)
d_k_demo = 8  # 하나의 attention head가 담당하는 차원 수라고 생각합시다
rope_demo = RoPE(d_model=d_k_demo, max_len=200)

# 서로 다른 두 "내용 벡터"를 하나는 Query, 하나는 Key라고 생각합니다.
# (실제로는 토큰마다 다른 Q, K가 있겠지만, 여기서는 "내용은 고정"한 채 "위치만" 바꿔보며
#  attention score가 어떻게 변하는지에만 집중하기 위해 q, k 내용을 고정합니다.)
q_content = torch.randn(d_k_demo)
k_content = torch.randn(d_k_demo)

def rope_attention_score(q, k, pos_q, pos_k, rope_module):
    q_rotated = rope_module.apply_rotation(q, torch.tensor([pos_q]))[0]
    k_rotated = rope_module.apply_rotation(k, torch.tensor([pos_k]))[0]
    return (q_rotated * k_rotated).sum().item()

print("거리(=position 차이)가 3으로 같은 세 가지 경우를 비교해봅니다:\n")
for pos_q, pos_k in [(2, 5), (10, 13), (50, 53)]:
    score = rope_attention_score(q_content, k_content, pos_q, pos_k, rope_demo)
    print(f"  query 위치={pos_q:>3}, key 위치={pos_k:>3} (거리=3)  ->  score = {score:.4f}")

print("\n절대 위치는 전혀 다른데도(2,5 / 10,13 / 50,53) score 값이 거의 동일합니다!")
print("(아주 미세한 오차는 부동소수점 연산 때문이며, 수학적으로는 정확히 같은 값입니다)")

print("\n반대로 거리가 다르면 score도 달라집니다:")
for pos_q, pos_k in [(2, 5), (2, 6), (2, 9)]:
    score = rope_attention_score(q_content, k_content, pos_q, pos_k, rope_demo)
    distance = pos_k - pos_q
    print(f"  query 위치={pos_q}, key 위치={pos_k} (거리={distance})  ->  score = {score:.4f}")

지금까지는 벡터 하나를 회전시키는 것만 봤습니다. 이제 이것을 실제 Self-Attention 레이어 안에 넣어서, Q와 K에 회전을 적용하는 완전한 `RoPEAttention` 모듈을 만들어봅니다.

V에는 회전을 적용하지 **않는다**는 점에 주의하세요 — 회전은 "어디서 왔는지(위치)"를 점수 계산에만 반영하기 위한 것이지, 실제로 전달되는 내용(Value)을 바꾸려는 게 아니기 때문입니다.

In [ ]:
class RoPEAttention(nn.Module):
    """RoPE를 Query, Key에 적용하는 Multi-Head Self-Attention"""

    def __init__(self, n_heads, d_model, max_len=5000):
        super().__init__()
        assert d_model % n_heads == 0, "d_model은 n_heads로 나누어 떨어져야 합니다."
        self.n_heads = n_heads
        self.d_model = d_model
        self.d_k = d_model // n_heads  # head 하나가 담당하는 차원 수

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        # 회전은 head 하나의 차원(d_k) 단위로 적용합니다.
        self.rope = RoPE(self.d_k, max_len=max_len)

    def forward(self, x):
        B, N, D = x.shape  # (batch, seq_len, d_model)

        # 입력을 Q, K, V로 투영한 뒤, head별로 나눕니다.
        Q = self.q_proj(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)  # (B, h, N, d_k)
        K = self.k_proj(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.n_heads, self.d_k).transpose(1, 2)

        # ── RoPE 핵심: Q, K에만 위치 기반 회전을 적용합니다 (V는 그대로 둡니다) ──
        Q = self.rope(Q)
        K = self.rope(K)

        # 회전된 Q, K로 평소와 동일하게 attention score, softmax, weighted sum을 계산합니다.
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, h, N, N)
        attn = torch.softmax(scores, dim=-1)
        output = torch.matmul(attn, V)  # (B, h, N, d_k)

        # head들을 다시 합쳐서 (B, N, D) 형태로 복원
        return output.transpose(1, 2).contiguous().view(B, N, D)

In [ ]:
rope_attn = RoPEAttention(n_heads=n_heads, d_model=d_model)
sample_input = torch.randn(2, 10, d_model)
sample_output = rope_attn(sample_input)
print("RoPEAttention 입력 shape :", sample_input.shape)
print("RoPEAttention 출력 shape :", sample_output.shape, " # 입력과 동일한 shape 유지")

### 참고: RoPE의 길이 외삽을 더 잘하게 만드는 기법들

RoPE는 수식 기반이라 Sinusoidal처럼 "원리적으로는" 학습 때보다 긴 문장에도 적용할 수 있습니다. 하지만 실제로는 한 번도 본 적 없는 긴 거리의 회전 각도를 만나면 성능이 떨어지는 경우가 많아서, 이를 보완하는 기법들이 따로 연구되었습니다. 지금 단계에서는 "이런 게 있다" 정도만 알아두면 충분합니다.

- **Position Interpolation**: 위치 인덱스 자체를 `(학습 길이 / 평가 길이)` 비율로 압축해서, 모델이 "익숙한 범위"의 각도만 보도록 만드는 방법
- **NTK-aware Scaling**: 위치를 압축하는 대신, 회전 속도를 정하는 기준값(`base`, 코드의 `10000`)을 조정해서 비슷한 효과를 내는 방법

두 방법 모두 핵심은 "학습 때 본 적 없는 회전 각도를 최대한 줄여서, 모델이 외삽 구간에서도 익숙한 패턴을 보게 만든다"는 것입니다. 더 깊이 알아보고 싶다면 각 기법의 원 논문이나 블로그를 검색해보는 것을 추천합니다.

## Section 4 (보너스). ALiBi (Attention with Linear Biases)

ALiBi는 세 방법과는 또 다른, 어떤 의미에서는 **가장 단순한** 접근입니다. (논문: Press et al., 2021)

### 핵심 아이디어
**Q, K, V 어디에도 손대지 않고, Attention score를 구한 뒤에 "거리에 비례하는 페널티(벌점)"를 그냥 더해버린다.**

$$\text{score}(i, j) = \frac{q_i \cdot k_j}{\sqrt{d_k}} \;-\; m \times (i - j) \qquad (j \le i,\ \text{causal})$$

여기서 `m`은 head마다 다르게 정해지는 **기울기(slope)** 상수이고, `(i-j)`는 query 위치 `i`와 key 위치 `j` 사이의 거리입니다. 거리가 멀수록 빼는 값이 커지므로, **"가까운 토큰일수록 더 잘 보이고, 먼 토큰일수록 점점 안 보이게"** 만드는 효과를 냅니다 (당연히 미래 위치, 즉 `j > i`인 부분은 다른 방법들처럼 causal mask로 아예 `-∞`로 막아둡니다).

머리(head)마다 기울기 `m`이 다른 이유는, 어떤 head는 가까운 단어 위주로 좁게 보고(기울기를 크게), 어떤 head는 멀리까지 폭넓게 보도록(기울기를 작게) 역할을 나누기 위해서입니다. 논문에서는 다음 공식으로 head별 기울기를 정합니다.

$$m_h = 2^{-\frac{8h}{n_{heads}}} \qquad (h = 1, 2, \ldots, n_{heads})$$

위치 인코딩을 위한 별도 벡터나 학습 파라미터가 전혀 없다는 점이 ALiBi의 가장 큰 특징입니다. 그냥 "거리표"를 score에 더하는 것뿐이라, 어떤 길이의 문장이 와도 그 자리에서 거리표를 계산하면 그만입니다. 그래서 길이 외삽 성능이 특히 좋다고 알려져 있습니다.

> ⚠️ **자주 하는 실수**: 이 거리표를 코드로 만들 때 "query 위치 - key 위치"와 "key 위치 - query 위치"를 헷갈리기 쉽습니다. 부호를 반대로 쓰면 causal mask 방향이 뒤집혀서, 모델이 미래 토큰을 보고 과거 토큰을 못 보는 정반대 상황이 벌어집니다. 아래 코드의 주석을 따라가며 부호를 꼭 확인해보세요.

In [ ]:
class ALiBiAttention(nn.Module):
    """ALiBi: Attention with Linear Biases — 위치 임베딩 없이, attention score에 거리 페널티만 더하는 방식"""

    def __init__(self, n_heads, d_model):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model

        # head별 기울기(slope) 계산: m_h = 2^(-8h/n_heads), h = 1, ..., n_heads
        # → h가 커질수록(뒤쪽 head일수록) 기울기가 기하급수적으로 작아짐
        slopes = 2 ** (-8 / n_heads * torch.arange(1, n_heads + 1).float())
        self.register_buffer('slopes', slopes)  # shape: (n_heads,)

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, N, D = x.shape
        d_k = D // self.n_heads

        Q = self.q_proj(x).view(B, N, self.n_heads, d_k).transpose(1, 2)  # (B, h, N, d_k)
        K = self.k_proj(x).view(B, N, self.n_heads, d_k).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.n_heads, d_k).transpose(1, 2)

        # 평소와 동일한 attention score
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (B, h, N, N)

        # ── ALiBi 거리표 만들기 ──
        positions = torch.arange(N, device=x.device)

        # distance[i, j] = i - j  →  "query가 key보다 얼마나 미래에 있는가" = 과거를 돌아보는 거리
        # (양수: key가 과거, 0: 자기 자신, 음수: key가 미래 → causal에서는 음수는 막아야 함)
        distance = positions.unsqueeze(1) - positions.unsqueeze(0)  # (N, N)

        causal_mask = (distance >= 0).float()  # j <= i 인 경우만 1 (과거+현재만 허용)

        # bias[i,j] = -slope * distance  → 거리가 멀수록(distance가 클수록) 더 큰 음수 페널티
        # slopes: (n_heads,) → (n_heads, 1, 1)로 모양을 맞춰서 head마다 다른 기울기를 곱함
        alibi_bias = -self.slopes.view(-1, 1, 1) * distance.float().unsqueeze(0)  # (n_heads, N, N)

        # 미래 위치(j > i)는 -1e9로 사실상 -무한대 처리하여 softmax 후 거의 0이 되도록 함
        alibi_bias = alibi_bias * causal_mask.unsqueeze(0) + (1 - causal_mask.unsqueeze(0)) * (-1e9)

        scores = scores + alibi_bias  # (B, h, N, N) + (h, N, N) → 브로드캐스팅
        attn = torch.softmax(scores, dim=-1)
        output = torch.matmul(attn, V)
        return output.transpose(1, 2).contiguous().view(B, N, D)

In [ ]:
alibi = ALiBiAttention(n_heads=n_heads, d_model=d_model)
print("head별 기울기(slope):", [round(s, 6) for s in alibi.slopes.tolist()])
print("→ 뒤쪽 head로 갈수록 기울기가 작아져서, '더 멀리까지 부드럽게' 보게 됩니다.\n")

# 거리표(distance)와 최종 bias가 실제로 어떻게 생겼는지 N=8짜리 작은 예시로 시각화해봅니다.
N_demo = 8
positions = torch.arange(N_demo)
distance = positions.unsqueeze(1) - positions.unsqueeze(0)
causal_mask = (distance >= 0).float()
bias_head0 = -alibi.slopes[0] * distance.float()
bias_head0_masked = bias_head0 * causal_mask + (1 - causal_mask) * (-1e9)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

im0 = axes[0].imshow(distance.numpy(), cmap='viridis')
axes[0].set_title('distance[i, j] = i - j')
axes[0].set_xlabel('key position (j)')
axes[0].set_ylabel('query position (i)')
plt.colorbar(im0, ax=axes[0])

# -1e9 부분은 그래프에서 너무 튀어 보이므로, 시각화를 위해 적당한 값으로 잘라서 보여줍니다.
# (잘라내는 값을 너무 크게(-10 등) 잡으면 정작 보고 싶은 "과거 영역의 그라데이션"이 한쪽으로 뭉쳐 보이므로,
#  실제 페널티 최대값(-slope*최대거리) 근처로 잘라야 색 대비가 잘 보입니다)
clip_value = -(alibi.slopes[0].item() * (N_demo - 1)) * 1.5
display_bias = bias_head0_masked.clamp(min=clip_value).numpy()
im1 = axes[1].imshow(display_bias, cmap='viridis')
axes[1].set_title('ALiBi bias (head 0), clipped for display')
axes[1].set_xlabel('key position (j)')
axes[1].set_ylabel('query position (i)')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

print("왼쪽: 대각선(i=j) 위쪽(미래, j>i)은 음수, 아래쪽(과거, j<i)은 양수인 순수 거리표입니다.")
print(f"오른쪽: 대각선 기준 위쪽(미래)은 매우 작은 값({clip_value:.2f} 이하, 거의 -무한대)으로 막혀 있고,")
print("        아래쪽(과거)은 거리가 멀어질수록 점점 더 작은(어두운) 페널티 값을 가집니다.")

### ALiBi 요약
- ✅ 추가 학습 파라미터가 전혀 없음 (head별 기울기는 고정된 수식으로 계산)
- ✅ 길이 외삽 성능이 특히 우수하다고 보고됨 (어떤 길이든 그 자리에서 거리표만 계산하면 됨)
- ✅ Q, K, V를 건드리지 않고 score에만 더하므로 구현이 비교적 단순함
- ⚠️ "거리"라는 하나의 신호만 사용하므로, RoPE처럼 더 풍부한 위치 표현(여러 주파수의 회전)에 비해 표현력이 제한적일 수 있음

## Section 5. 종합 비교 실험: 학습보다 긴 문장(외삽)에서도 잘 동작할까?

실제 서비스에서는 이런 상황이 자주 벌어집니다. **모델을 짧은 문장(예: 64토큰)으로 학습시켰는데, 실제 사용자는 더 긴 문장(예: 128토큰)을 입력**합니다. 이렇게 "학습 때 본 적 없는 길이"를 다루는 능력을 **길이 외삽(length extrapolation)**이라고 부릅니다.

지금까지 만든 네 가지 방법(Sinusoidal, Learned, RoPE, ALiBi)을 모두 같은 조건에서 비교해봅니다.

- `train_len = 64`: "학습 때 사용했다"고 가정하는 시퀀스 길이
- `eval_len = 128`: 학습보다 2배 긴, "실제 사용 시" 시퀀스 길이

각 방법에 두 길이를 모두 입력해보고, 에러 없이 동작하는지(shape가 잘 나오는지) 확인합니다. 이 실험은 **"수식·구조적으로 더 긴 입력을 받아낼 수 있는가"**를 확인하는 것이지, "성능이 얼마나 좋은가"까지 검증하는 것은 아니라는 점에 유의하세요 — 실제 성능 저하 여부는 학습된 모델로 별도 평가가 필요합니다.

In [ ]:
print("=== 위치 인코딩 비교 실험 ===\n")

d_model = 128
n_heads = 4
train_len = 64   # 학습 시 시퀀스 길이라고 가정
eval_len = 128   # 평가 시 시퀀스 길이 (2배 외삽)

# 더미 입력 (실제 토큰 임베딩이 아니라, shape과 동작만 확인하기 위한 무작위 값)
x_train = torch.randn(2, train_len, d_model)
x_eval = torch.randn(2, eval_len, d_model)

results = {}  # 나중에 표로 정리하기 위해 결과를 저장해둘 dict

# 1. Sinusoidal -----------------------------------------------------------
sin_pe = SinusoidalPE(d_model, max_len=5000)
out_train = sin_pe(x_train)
try:
    out_eval = sin_pe(x_eval)  # 수식 기반이라 max_len(5000) 이내면 항상 계산 가능
    print(f"Sinusoidal : 학습 {tuple(out_train.shape)} -> 평가 {tuple(out_eval.shape)}  ✅ 외삽 가능")
    results['Sinusoidal'] = True
except Exception as e:
    print(f"Sinusoidal : 학습 {tuple(out_train.shape)} -> 평가 실패 ❌ ({e})")
    results['Sinusoidal'] = False

# 2. Learned (max_len=100, 즉 eval_len=128보다 작게 일부러 설정) -----------
learned_pe = LearnedPE(d_model, max_len=100)
out_train = learned_pe(x_train)
try:
    out_eval = learned_pe(x_eval)  # max_len(100) < eval_len(128) → 에러 발생 예상
    print(f"Learned    : 학습 {tuple(out_train.shape)} -> 평가 {tuple(out_eval.shape)}  ✅ 외삽 가능")
    results['Learned'] = True
except Exception as e:
    print(f"Learned    : 학습 {tuple(out_train.shape)} -> 평가 실패 ❌ (max_len 초과로 위치 임베딩 테이블에 해당 위치가 없음)")
    results['Learned'] = False

# 3. RoPE -------------------------------------------------------------------
rope_attn = RoPEAttention(n_heads, d_model, max_len=5000)
out_train = rope_attn(x_train)
try:
    out_eval = rope_attn(x_eval)  # 수식 기반이라 max_len(5000) 이내면 항상 계산 가능
    print(f"RoPE       : 학습 {tuple(out_train.shape)} -> 평가 {tuple(out_eval.shape)}  ✅ 외삽 가능")
    results['RoPE'] = True
except Exception as e:
    print(f"RoPE       : 학습 {tuple(out_train.shape)} -> 평가 실패 ❌ ({e})")
    results['RoPE'] = False

# 4. ALiBi --------------------------------------------------------------------
alibi_attn = ALiBiAttention(n_heads, d_model)
out_train = alibi_attn(x_train)
try:
    out_eval = alibi_attn(x_eval)  # 거리표를 그 자리에서 계산하므로 길이 제한이 없음
    print(f"ALiBi      : 학습 {tuple(out_train.shape)} -> 평가 {tuple(out_eval.shape)}  ✅ 외삽 가능")
    results['ALiBi'] = True
except Exception as e:
    print(f"ALiBi      : 학습 {tuple(out_train.shape)} -> 평가 실패 ❌ ({e})")
    results['ALiBi'] = False

In [ ]:
summary = pd.DataFrame({
    '방법': ['Sinusoidal', 'Learned', 'RoPE', 'ALiBi'],
    '위치 정보 적용 방식': ['임베딩에 더하기 (절대)', '임베딩에 더하기 (절대)', 'Q,K 회전 (상대)', 'score에 거리 페널티 (상대)'],
    '학습 파라미터': ['없음', f'{100 * d_model:,}개', '없음', '없음'],
    f'길이 {eval_len} 외삽': [
        '가능' if results['Sinusoidal'] else '불가능',
        '가능' if results['Learned'] else '불가능',
        '가능' if results['RoPE'] else '불가능',
        '가능' if results['ALiBi'] else '불가능',
    ],
})
summary

## 정리하며

| 방법 | 한 줄 요약 | 대표 사용처 |
|---|---|---|
| **Sinusoidal** | 고정 수식으로 만든 절대 위치 벡터를 더한다 | 원조 Transformer |
| **Learned** | 위치마다 학습되는 벡터를 더한다 (구현은 가장 쉬움) | BERT, GPT-2 |
| **RoPE** | Q, K를 위치에 비례한 각도로 회전시켜 상대 거리를 score에 자연스럽게 녹인다 | LLaMA, GPT-NeoX, PaLM 등 최신 LLM 다수 |
| **ALiBi** | Q,K,V는 그대로 두고, score에 거리 비례 페널티만 더한다 (가장 단순) | BLOOM 등 |

### 핵심으로 기억할 것
1. Self-Attention은 원래 순서를 모릅니다 — 그래서 위치 정보를 **반드시** 어딘가에 주입해야 합니다. (Section 0)
2. 위치 정보를 **"더할 것이냐(절대 위치, Sinusoidal/Learned)" vs "Q,K를 변형할 것이냐(상대 위치, RoPE/ALiBi)"**가 가장 큰 갈림길입니다.
3. **학습 파라미터가 없는 방법(Sinusoidal, RoPE, ALiBi)일수록 일반적으로 길이 외삽에 유리**하고, **학습 파라미터가 있는 방법(Learned)은 유연하지만 정해진 길이를 벗어나면 동작 자체가 불가능**합니다.
4. 최신 LLM 다수가 RoPE를 채택한 이유는, 상대 위치를 자연스럽게 표현하면서도 학습 파라미터가 없어 비교적 외삽에도 유리하기 때문입니다.

### 직접 더 해보면 좋은 실험
- `d_model`, `n_heads`, `train_len`/`eval_len` 값을 바꿔가며 각 방법의 동작을 다시 확인해보세요.
- RoPE의 `base` 값(기본 10000)을 바꾸면 회전 속도가 어떻게 달라지는지 Section 3의 시각화 코드를 재사용해서 확인해보세요.
- 실제로 작은 데이터셋(예: 간단한 시퀀스 정렬/복사 태스크)으로 4가지 방법을 각각 학습시켜, "외삽 가능 여부"가 실제 성능 차이로 이어지는지 검증해보세요.